In [ ]:
!pip install -q openai-whisper
!pip install -q transformers accelerate
!pip install -q torch torchvision
!pip install -q opencv-python
!pip install -q groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 24.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 6.1 MB/s eta 0:00:00


In [ ]:
!pip install faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 108.5 MB/s eta 0:00:00


In [ ]:
import os
os.environ["GROQ_API_KEY"] = "GROQ_KEY"
os.environ["GEMINI_API_KEY"] = "GEMINI_KEY"


In [ ]:
from faster_whisper import WhisperModel
import torch
import cv2
import numpy as np
import json
from dataclasses import dataclass
from typing import List, Dict
from transformers import CLIPProcessor, CLIPModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONFIDENCE_THRESHOLD = 0.85
FRAME_INTERVAL_SEC = 1
TOP_K_FRAMES = 4

In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timer(name):
    start = time.time()
    yield
    print(f"⏱ {name}: {time.time() - start:.2f}s")


In [ ]:
whisper_model = WhisperModel("base", device="cuda", compute_type="float16")

In [ ]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip_model = clip_model.half()
clip_model.eval()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05,

In [ ]:
def transcribe_video(video_path):

    # Faster-Whisper returns a tuple: (iterable_segments, transcription_info)
    segments, info = whisper_model.transcribe(video_path)

    segments_out = []
    full_text = ""

    # Iterate directly over the segments generator
    for seg in segments:
        full_text += seg.text + " "
        segments_out.append({
            "start": seg.start,
            "end": seg.end,
            "text": seg.text.strip()
        })

    return segments_out, full_text.strip()

In [ ]:
def full_text(video_path):
    segments, info = whisper_model.transcribe(video_path)

    full_text = ""
    structured_segments = []

    for seg in segments:
        structured_segments.append({
            "start": seg.start,
            "end": seg.end,
            "text": seg.text.strip()
        })
        full_text += seg.text.strip() + " "

    print("\n===== WHISPER TRANSCRIPT =====\n")
    print(full_text.strip())
    print("\n==============================\n")

    return structured_segments, full_text.strip()

In [ ]:
transcript,_ = full_text("/content/test6.mp4")


===== WHISPER TRANSCRIPT =====

Doing an inspection of sprinkler system, are sprinklers heads located at the top and bottom of the stairways? Yes, they are located. Is the annual inspection and testing of the sprinkler system up to date? Yes, it is up to date. Is the annual inspection and testing of standpipes up to date? Yes, it is up to date. Is the fire hydrant located with a hundred feet of fire department connection? Fire hydrant, yes. Are signs of sprinkler system up to date or standpipes visible? Yes, visible. Is there a backflow prevented or check wall at the system? Yes, it is visible. Is the sprinkler head located at the bottom of the elevator pit within two feet from the floor? No, is the attic sprinkler heard? No. Are sprinklers head installed under obstructions over four feet wide? No. Are sprinkler heads located in all mechanical communication electrical storage rooms? No. Are dry pipe systems? Is the trip test connection located at the end of the most distance sprinkler

In [ ]:
print(transcript)

[{'start': 0.0, 'end': 10.0, 'text': 'Doing an inspection of sprinkler system, are sprinklers heads located at the top and bottom of the stairways?'}, {'start': 10.0, 'end': 13.0, 'text': 'Yes, they are located.'}, {'start': 13.0, 'end': 17.0, 'text': 'Is the annual inspection and testing of the sprinkler system up to date?'}, {'start': 17.0, 'end': 20.0, 'text': 'Yes, it is up to date.'}, {'start': 20.0, 'end': 24.0, 'text': 'Is the annual inspection and testing of standpipes up to date?'}, {'start': 24.0, 'end': 27.0, 'text': 'Yes, it is up to date.'}, {'start': 27.0, 'end': 31.0, 'text': 'Is the fire hydrant located with a hundred feet of fire department connection?'}, {'start': 31.0, 'end': 34.0, 'text': 'Fire hydrant, yes.'}, {'start': 34.0, 'end': 38.0, 'text': 'Are signs of sprinkler system up to date or standpipes visible?'}, {'start': 38.0, 'end': 41.0, 'text': 'Yes, visible.'}, {'start': 41.0, 'end': 44.0, 'text': 'Is there a backflow prevented or check wall at the system?'},

In [ ]:
def build_context(segments, idx, window=3):
    start = max(0, idx - (window - 1))
    context_segments = segments[start:idx+1]
    text = " ".join([s["text"] for s in context_segments])
    start_time = context_segments[0]["start"]
    end_time = context_segments[-1]["end"]
    return text.strip(), start_time, end_time


In [ ]:
from groq import Groq

client = Groq()
# print(client.models.list())

def call_llm(prompt: str) -> str:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # UPDATED MODEL
        messages=[
            {
                "role": "system",
                "content": "You are a strict JSON extraction engine. Output ONLY valid JSON."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.0,
        max_tokens=512,
        top_p=0.9
    )

    return response.choices[0].message.content.strip()


In [ ]:
import json
import time

def generate_transcript_variants(base_transcript, n=20):
    variants = []

    for i in range(n):
        prompt = f"""
You are generating realistic fire inspection narration.

Rewrite the following transcript into a DIFFERENT but
semantically equivalent spoken inspection narration.

Rules:
- You don't need to Keep the same checklist decisions
- Change wording, sentence order, phrasing
- You can also change the YES/NO answers from the transcript
- You can also change the checklist responses from the transcript and play with the order of things
- Natural spoken English
- One paragraph only
- Do NOT add or remove checklist facts

Transcript:
\"\"\"{base_transcript}\"\"\"

Return ONLY the rewritten transcript text.
"""

        raw = call_llm(prompt).strip()

        # Basic sanity check
        if len(raw) < 30:
            print(f"⚠️ Skipped short output at iteration {i}")
            continue
        print(raw)

        variants.append(raw)

        time.sleep(0.3)  # avoids rate / quality collapse

    return variants

In [ ]:
variants = generate_transcript_variants(transcript, n=20)

with open("derived_transcripts.json", "w") as f:
    json.dump(variants, f, indent=2)

{
  "narration": "Alright, conducting a sprinkler system inspection here. I can confirm that the annual inspection and testing for both the sprinkler system and standpipes are up to date. The fire hydrant is indeed located within one hundred feet of the fire department connection, and the sprinkler system and standpipe signs are clearly visible. Also, there's a backflow preventer or check valve present in the system. Sprinkler heads are located at the top and bottom of the stairways. However, the sprinkler head is NOT located at the bottom of the elevator pit within two feet from the floor, and there are no sprinkler heads installed under obstructions over four feet wide, nor are there sprinkler heads in all mechanical communication electrical storage rooms. Finally, for the dry pipe system, the trip test connection is NOT located at the end of the most distant sprinkler pipe in the upper story. That concludes this part of the inspection."
}
{
  "narration": "Alright, conducting a spri

In [ ]:
from google import genai
from google.genai import types

# Setup - The new way (google-genai 1.0+)
client = genai.Client(api_key="GEMINI_KEY")

def call_llm(prompt: str) -> str:
    response = client.models.generate_content(
        model="gemini-2.0-flash", # Use the latest Flash model
        config=types.GenerateContentConfig(
            system_instruction="You are a strict JSON extraction engine. Output ONLY valid JSON.",
            response_mime_type="application/json",
            temperature=0.0,
            max_output_tokens=512,
        ),
        contents=prompt
    )

    return response.text.strip()

In [ ]:
def safe_extract(prompt):
    raw = call_llm(prompt)

    try:
        parsed = json.loads(raw)

        # If model returns list, extract first element
        if isinstance(parsed, list) and len(parsed) > 0:
            parsed = parsed[0]

        # Ensure dictionary format
        if not isinstance(parsed, dict):
            raise ValueError("Not dict")

        return parsed

    except:
        # Retry once
        retry_prompt = prompt + "\nREMINDER: Output ONLY a single JSON object."
        raw_retry = call_llm(retry_prompt)

        try:
            parsed = json.loads(raw_retry)

            if isinstance(parsed, list) and len(parsed) > 0:
                parsed = parsed[0]

            if not isinstance(parsed, dict):
                raise ValueError("Still not dict")

            return parsed

        except:
            return {
                "item_id": None,
                "decision": None,
                "reasoning": "",
                "confidence": 0.0
            }


In [ ]:
CHECKLIST_DESCRIPTION = """
App 4 — Sprinkler Systems (NFPA 13 / NFPA 14)

Checklist Items:
1 = Sprinkler heads at top and bottom of stairways
2 = Annual inspection & testing of sprinkler system (date required)
3 = Annual inspection & testing of standpipes (date required)
4 = Fire hydrant within 100 ft of fire department connection
5 = FDC / standpipe signage visible
6 = Backflow preventer or check valve present
7 = Sprinkler head in elevator pit within 2 ft of floor
8 = Attic sprinklered
9 = Sprinkler heads under fixed obstructions >4 ft
10 = Sprinklers in all mechanical / electrical / storage rooms
11 = Dry pipe system trip test connection at most distant pipe
"""

def extract_all_decisions(full_transcript: str):
    prompt = f"""
You are a STRICT structured extraction engine for fire sprinkler inspections.

Your task:
From the inspector transcript, extract FINAL checklist decisions for App 4.

STRICT RULES (VERY IMPORTANT):
- Extract ONLY if a clear final decision is explicitly stated.
- If vague, unclear, implied, or conversational → decision = null.
- If conflicting statements appear → decision = null.
- Decision must be EXACTLY one of: YES, NO, N/A, or null.
- Confidence must reflect clarity (0.0 to 1.0).
- Do NOT infer. Do NOT assume. Do NOT guess.

DATE RULES (Items 2 & 3 ONLY):
- If a date is spoken, extract it EXACTLY as mentioned.
- If compliant but no date spoken → date = "N/A".
- If non-compliant → date = null.

{CHECKLIST_DESCRIPTION}

Transcript:
\"\"\"{full_transcript}\"\"\"


OUTPUT FORMAT:
Return EXACTLY 11 JSON objects (item_id 1–11) in a JSON array.

For items [1,4,5,6,7,8,9,10,11]:
{{
  "item_id": int,
  "decision": "YES" | "NO" | "N/A" | null,
  "reasoning": "exact phrase from transcript or empty string",
  "confidence": float,
  "date": null
}}

For items [2,3]:
{{
  "item_id": int,
  "decision": "YES" | "NO" | null,
  "reasoning": "exact phrase from transcript or empty string",
  "confidence": float,
  "date": "date string" | "N/A" | null
}}

If an item is NOT clearly mentioned:
{{
  "item_id": <that item number>,
  "decision": null,
  "reasoning": "",
  "confidence": 0.0,
  "date": null
}}

Respond ONLY with VALID JSON. No explanations. No markdown.
"""

    raw = call_llm(prompt)

    # ---------------- Robust JSON normalization ----------------
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"\[.*\]", raw, re.S)
        if not match:
            return []
        parsed = json.loads(match.group())

    # Handle list of JSON strings (LLM weirdness)
    if isinstance(parsed, list) and parsed and isinstance(parsed[0], str):
        parsed = [json.loads(x) for x in parsed]

    # Final validation & cleanup
    clean = []
    for item in parsed:
        if not isinstance(item, dict):
            continue
        if "item_id" not in item:
            continue
        clean.append(item)

    return clean


In [ ]:
APP4_ITEM_MAP = {
    1: "Sprinkler heads at top and bottom of stairways",
    2: "Annual inspection & testing of sprinkler system",
    3: "Annual inspection & testing of standpipes",
    4: "Fire hydrant within 100 ft of fire department connection",
    5: "FDC / standpipe signage visible",
    6: "Backflow preventer or check valve present",
    7: "Sprinkler head in elevator pit within 2 ft of floor",
    8: "Attic sprinklered",
    9: "Sprinkler heads under fixed obstructions over 4 ft wide",
    10: "Sprinklers in all mechanical / electrical / storage rooms",
    11: "Dry pipe system trip test connection at most distant pipe"
}

DATE_ITEMS = {2, 3}


In [ ]:
def extract_decision_for_item(full_transcript: str, item_id: int):

    item_desc = APP4_ITEM_MAP[item_id]
    is_date_item = item_id in DATE_ITEMS

    date_block = ""
    if is_date_item:
        date_block = """
DATE RULES:
- If compliant and a date is spoken, extract it EXACTLY.
- If compliant but no date spoken → date = "N/A".
- If non-compliant → date = null.
"""

    prompt = f"""
You are a STRICT fire inspection extraction engine.

Your task:
Determine the FINAL decision for ONE checklist item ONLY.

Checklist Item {item_id}:
"{item_desc}"

STRICT RULES:
- Extract ONLY if the inspector makes a clear final decision.
- Do NOT infer. Do NOT assume.
- If unclear, conflicting, or not explicitly mentioned → decision = null.
- Decision must be exactly: YES, NO, or N/A (only if logically applicable).
- Confidence reflects clarity (0.0–1.0).

DECISION INTERPRETATION RULES (CRITICAL):

The inspector speaks in natural language.
You MUST interpret final intent, not just keywords.

Map spoken phrases to decisions as follows:

YES if the inspector says or implies compliance, including phrases like:
- "yes"
- "is current"
- "up to date"
- "acceptable"
- "looks good"
- "is present"
- "is visible"
- "complies"
- "meets requirement"

NO if the inspector states non-compliance, including phrases like:
- "no"
- "not present"
- "missing"
- "not located"
- "does not meet"
- "not within required distance"

N/A if the inspector explicitly states:
- "not applicable"
- "does not apply"
- "system type does not require this"
- "not required for this system"

If a system condition makes an item irrelevant (e.g., dry pipe test on a wet system),
you MUST return decision = "N/A".

{date_block}

Transcript:
\"\"\"{full_transcript}\"\"\"

Respond ONLY with valid JSON:

{{
  "item_id": {item_id},
  "decision": "YES" | "NO" | "N/A" | null,
  "reasoning": "exact quoted phrase from transcript or empty string",
  "confidence": float,
  "date": {"\"date string\" | \"N/A\" | null" if is_date_item else "null"}
}}
"""

    result = safe_extract(prompt)

    # ❗ DO NOT fabricate timestamps
    result["start_time"] = None
    result["end_time"] = None

    return result


In [ ]:
def validate_decision(context_text, decision_data):
    if decision_data["decision"] is None:
        return False, 0.0

    prompt = f"""
You are a verification engine.

Transcript:
\"\"\"{context_text}\"\"\"

Extracted decision:
Item: {decision_data["item_id"]}
Decision: {decision_data["decision"]}
Reasoning: {decision_data["reasoning"]}

Question:
Is this decision clearly and explicitly supported by the transcript?

Respond ONLY with JSON:
{{
  "valid": true or false,
  "confidence": float (0.0 to 1.0)
}}
"""

    result = safe_extract(prompt)

    return result.get("valid", False), result.get("confidence", 0.0)


In [ ]:
def detect_contradiction(context_text):
    prompt = f"""
Does the transcript below contain conflicting inspection conclusions?

Transcript:
\"\"\"{context_text}\"\"\"

Respond ONLY JSON:
{{
  "contradiction": true or false
}}
"""
    result = safe_extract(prompt)
    return result.get("contradiction", False)


In [ ]:
def detect_ambiguity(context_text):
    prompt = f"""
Is the inspector statement vague or non-explicit?

Transcript:
\"\"\"{context_text}\"\"\"

Respond ONLY JSON:
{{
  "ambiguous": true or false
}}
"""
    result = safe_extract(prompt)
    return result.get("ambiguous", False)


In [ ]:
def compute_final_confidence(extract_conf, validate_conf, clip_sim):
    return min(extract_conf, validate_conf, clip_sim)


In [ ]:
def robust_decision(decision_data):

    if decision_data["decision"] is None:
        return "NEEDS_REVIEW", 0.0

    reasoning = decision_data.get("reasoning", "").lower()

    # Hard safety override for damage
    if any(word in reasoning for word in ["damage", "corrosion", "rust", "leak"]):
        return "NO", decision_data.get("confidence", 0.9)

    return decision_data["decision"], decision_data.get("confidence", 0.9)


In [ ]:
import cv2

def center_crop(frame, crop_ratio=0.6):
    """
    Center-crop a frame.

    Args:
        frame (np.ndarray): BGR image
        crop_ratio (float): fraction of frame to keep (0.5–0.7 recommended)

    Returns:
        np.ndarray: center-cropped frame
    """
    h, w = frame.shape[:2]

    ch = int(h * crop_ratio)
    cw = int(w * crop_ratio)

    y1 = max((h - ch) // 2, 0)
    x1 = max((w - cw) // 2, 0)

    return frame[y1:y1 + ch, x1:x1 + cw]


In [ ]:
def sample_frames(video_path, fps=1.0):
    import subprocess, json, numpy as np

    # Probe video dimensions
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=width,height", "-of", "json", video_path

    ]
    result = subprocess.run(cmd, capture_output=True, check=True)
    probe = json.loads(result.stdout)
    w = probe["streams"][0]["width"]
    h = probe["streams"][0]["height"]
    frame_size = w * h * 3  # RGB

    # FFmpeg GPU command
    ffmpeg_cmd = [
    "ffmpeg", "-loglevel", "error",
    "-hwaccel", "cuda",
    "-hwaccel_output_format", "cuda",  # Add this line
    "-c:v", "hevc_cuvid",
    "-i", video_path,
    "-vf", f"fps={fps},hwdownload,format=nv12,format=rgb24",
    "-f", "rawvideo", "-pix_fmt", "rgb24", "-"
]

    pipe = subprocess.Popen(ffmpeg_cmd, stdout=subprocess.PIPE)

    frames, timestamps = [], []
    idx = 0

    while True:
        raw = pipe.stdout.read(frame_size)
        if len(raw) != frame_size:
            break
        frame = np.frombuffer(raw, np.uint8).reshape((h, w, 3)).copy()
        frames.append(frame)
        timestamps.append(idx / fps)
        idx += 1

    pipe.stdout.close()
    pipe.wait()

    return frames, timestamps

In [ ]:
def get_frames_for_reasoning(start_time, end_time, frames, timestamps,
                             window=2, max_frames=3):
    """
    Returns multiple evidence frames around reasoning window.
    """

    start = max(0, start_time - window)
    end = end_time + window

    # Collect candidate frames in time window
    candidate_indices = [
        i for i, t in enumerate(timestamps)
        if start <= t <= end
    ]

    if not candidate_indices:
        return []

    # Optional: Sharpness scoring
    def laplacian_variance(frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        return cv2.Laplacian(gray, cv2.CV_64F).var()

    scored = [
        (laplacian_variance(frames[i]), i)
        for i in candidate_indices
    ]

    scored.sort(reverse=True)

    selected_indices = [idx for _, idx in scored[:max_frames]]

    return [
        {
            "frame": frames[i],
            "timestamp": timestamps[i]
        }
        for i in selected_indices
    ]


In [ ]:
def compute_frame_embeddings(frames, batch_size=16):
    device = clip_model.device
    all_embeddings = []

    for i in range(0, len(frames), batch_size):
        batch = frames[i:i+batch_size]

        inputs = clip_processor(
            images=batch,
            return_tensors="pt",
            padding=True
        ).to(device)

        with torch.no_grad():
            outputs = clip_model.get_image_features(**inputs)

            # CRITICAL FIX: The log shows 'pooler_output' is available
            # Try pooler_output first, fallback to the raw object if it's already a tensor
            if hasattr(outputs, 'pooler_output'):
                feats = outputs.pooler_output
            elif hasattr(outputs, 'image_embeds'):
                feats = outputs.image_embeds
            else:
                feats = outputs

            # Perform the normalization on the resulting tensor
            feats = feats / feats.norm(dim=-1, keepdim=True)

        all_embeddings.append(feats)

    return torch.cat(all_embeddings, dim=0)

In [ ]:
ENRICHED_QUERY_MAP = {

    # 1. Sprinkler heads at top and bottom of stairways
    1: [
        "sprinkler head installed near staircase ceiling",
        "sprinkler head at top of stairwell",
        "sprinkler head visible above stairs",
        "sprinkler head near bottom of stairway"
    ],

    # 2. Annual inspection & testing of sprinkler system (DATE → audio primary)
    2: [
        "sprinkler inspection tag with date",
        "sprinkler system inspection label",
        "paper tag attached to sprinkler riser",
        "inspection record label on sprinkler piping"
    ],

    # 3. Annual inspection & testing of standpipes (DATE → audio primary)
    3: [
        "standpipe inspection tag",
        "standpipe system label with date",
        "fire standpipe valve with inspection tag",
        "standpipe riser inspection record"
    ],

    # 4. Fire hydrant within 100 feet of FDC
    4: [
        "fire department connection outside building",
        "FDC connection on exterior wall",
        "fire hydrant near building exterior",
        "hydrant and fire department connection in same view"
    ],

    # 5. FDC / standpipe signage visible
    5: [
        "fire department connection sign",
        "red FDC sign mounted on wall",
        "standpipe sign near connection",
        "metal sign indicating fire department connection"
    ],

    # 6. Backflow preventer or check valve present
    6: [
        "backflow preventer on sprinkler pipe",
        "check valve installed on sprinkler system",
        "large valve assembly on fire sprinkler piping",
        "backflow prevention device near riser"
    ],

    # 7. Sprinkler head in elevator pit within 2 ft of floor
    7: [
        "sprinkler head inside elevator pit",
        "sprinkler head near elevator shaft bottom",
        "fire sprinkler in elevator pit",
        "sprinkler head close to floor in elevator shaft"
    ],

    # 8. Attic sprinklered
    8: [
        "sprinkler piping in attic",
        "sprinkler head installed in attic space",
        "fire sprinkler system above ceiling",
        "sprinkler pipe visible in attic"
    ],

    # 9. Sprinklers under fixed obstructions >4 ft
    9: [
        "sprinkler head below large duct",
        "sprinkler installed under ceiling obstruction",
        "sprinkler head beneath fixed obstruction",
        "sprinkler clearance under beams or ducts"
    ],

    # 10. Sprinklers in mechanical / electrical / storage rooms
    10: [
        "sprinkler head in electrical room",
        "sprinkler installed in mechanical room",
        "sprinkler head inside storage room",
        "fire sprinkler in utility room"
    ],

    # 11. Dry pipe system trip test connection at remote point
    11: [
        "dry pipe sprinkler system test connection",
        "trip test connection on dry pipe system",
        "sprinkler test valve at end of pipe",
        "dry pipe system inspector test connection"
    ]
}


In [ ]:
def build_query(item_id, reasoning_text, decision):

    enriched_terms = ENRICHED_QUERY_MAP.get(item_id, [])

    query = reasoning_text + " " + reasoning_text

    # Boost defect words if decision is NO
    if decision == "NO":
        query += " damaged rust corrosion crack broken"

    query += " " + " ".join(enriched_terms)

    return query


In [ ]:
def clip_text_embedding(text):

    inputs = clip_processor(
        text=[text],
        return_tensors="pt",
        padding=True
    ).to(DEVICE)

    # Run text encoder directly
    text_outputs = clip_model.text_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

    pooled_output = text_outputs.pooler_output

    text_features = clip_model.text_projection(pooled_output)

    # Normalize
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    return text_features[0].cpu()


In [ ]:
def retrieve_frames_hybrid(
    item_id,
    reasoning_text,
    decision,
    start_time,
    end_time,
    frame_embeddings,
    frames,
    timestamps,
    window=3,
    top_k=3
):
    device = frame_embeddings.device

    # 1️⃣ HARD STOP for missing time
    if start_time is None or end_time is None:
        # if item_id in [1, 5, 6, 8]:  # ❌ removed item 2
        #     return retrieve_frames_global(
        #         item_id, decision, frame_embeddings, frames, timestamps, top_k
        #     )
        return []

    # 2️⃣ Build query
    query = build_query(item_id, reasoning_text, decision)

    # 3️⃣ Safe window
    start = max(0.0, start_time - window)
    end = end_time + window

    candidate_indices = [
        i for i, t in enumerate(timestamps)
        if start <= t <= end
    ]

    if not candidate_indices:
        return []

    # 4️⃣ Text embedding (normalized + device-safe)
    text_embedding = clip_text_embedding(query).to(device)
    text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True)

    # 5️⃣ Similarity scoring
    scored = []
    for i in candidate_indices:
        img_emb = frame_embeddings[i]
        sim = torch.dot(img_emb, text_embedding).item()
        scored.append((sim, i))

    scored.sort(reverse=True)
    selected = [idx for _, idx in scored[:top_k]]

    return [{"frame": frames[i], "timestamp": timestamps[i]} for i in selected]


In [ ]:
def retrieve_frames_global(
    item_id,
    decision,
    frame_embeddings,
    frames,
    timestamps,
    top_k=3
):
    device = frame_embeddings.device

    query_text = " ".join(ENRICHED_QUERY_MAP[item_id])

    text_emb = clip_text_embedding(query_text).to(device)
    text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)

    scored = []
    for i in range(len(frame_embeddings)):
        sim = torch.dot(frame_embeddings[i], text_emb).item()
        scored.append((sim, i))

    scored.sort(reverse=True)

    return [
        {"timestamp": timestamps[idx]}
        for _, idx in scored[:top_k]
    ]


In [ ]:
def find_reasoning_time(reasoning_text, segments, min_overlap=0.4):
    if not reasoning_text:
        return None, None

    r_tokens = set(reasoning_text.lower().split())
    if not r_tokens:
        return None, None

    best_match = None
    best_score = 0.0

    for seg in segments:
        s_tokens = set(seg["text"].lower().split())
        if not s_tokens:
            continue

        overlap = len(r_tokens & s_tokens) / len(r_tokens)

        if overlap > best_score:
            best_score = overlap
            best_match = seg

    if best_match and best_score >= min_overlap:
        return best_match["start"], best_match["end"]

    return None, None


In [ ]:
def final_decision(decision_data, best_frame_similarity):
    if decision_data["confidence"] < CONFIDENCE_THRESHOLD:
        return "NEEDS_REVIEW", decision_data["confidence"]

    if best_frame_similarity < 0.23:  # Tune empirically
        return "NEEDS_REVIEW", min(decision_data["confidence"], best_frame_similarity)

    return decision_data["decision"], min(decision_data["confidence"], best_frame_similarity)


In [ ]:
def extract_all_decisions_itemwise(full_transcript: str):
    results = []

    for item_id in range(1, 12):  # App4 has 11 items
        decision = extract_decision_for_item(
            full_transcript=full_transcript,
            item_id=item_id
        )
        results.append(decision)

    return results


In [ ]:
def run_inspection_pipeline(video_path):

    with timer("Whisper"):
        segments, full_transcript = transcribe_video(video_path)

    with timer("Frame sampling"):
        frames, timestamps = sample_frames(video_path)

    with timer("CLIP embeddings"):
        frame_embeddings = compute_frame_embeddings(frames)

    with timer("LLM extraction"):
        all_decisions = extract_all_decisions_itemwise(full_transcript)

    checklist_results = []

    with timer("Evidence + logic"):
        for decision_data in all_decisions:

            item_id = decision_data["item_id"]

            start, end = find_reasoning_time(decision_data.get("reasoning", ""), segments)

            if start is None:
                if item_id in [5, 6, 8]:  # search-required items
                    evidence_frames = retrieve_frames_global(
                        item_id=item_id,
                        decision=decision_data.get("decision"),
                        frame_embeddings=frame_embeddings,
                        frames=frames,
                        timestamps=timestamps
                    )
                else:
                    evidence_frames = []
                    status = "NEEDS_REVIEW"
                    confidence = 0.0
                    # skip visual retrieval for this item
                    continue


            evidence_frames = retrieve_frames_hybrid(
                item_id=item_id,
                reasoning_text=decision_data.get("reasoning", ""),
                decision=decision_data.get("decision"),
                start_time=start,
                end_time=end,
                frame_embeddings=frame_embeddings,
                frames=frames,
                timestamps=timestamps
            )

            status, conf = robust_decision(decision_data)

            checklist_results.append({
                "item_id": item_id,
                "status": status,
                "comment": decision_data.get("reasoning", ""),
                "date": decision_data.get("date"),
                "evidence_frames": [
                    {"timestamp": f["timestamp"]} for f in evidence_frames
                ],
                "confidence_score": conf
            })

    print("✅ Inspection processing complete.")
    return checklist_results


In [ ]:
import matplotlib.pyplot as plt
def show_results_with_frames(results, video_path):
    frames, timestamps = sample_frames(video_path)

    import matplotlib.pyplot as plt

    for r in results:
        print("Item:", r["item_id"])
        print("Status:", r["status"])
        print("Confidence:", r["confidence_score"])
        print("Comment:", r["comment"])
        print()

        for ev in r.get("evidence_frames", []):
            ts = ev["timestamp"]

            closest_idx = min(
                range(len(timestamps)),
                key=lambda i: abs(timestamps[i] - ts)
            )

            plt.imshow(cv2.cvtColor(frames[closest_idx], cv2.COLOR_BGR2RGB))
            plt.title(f"Timestamp: {ts}")
            plt.axis("off")
            plt.show()


In [ ]:
def test_llm_stability(variants):
    outputs = []

    for i, t in enumerate(variants):
        decisions = extract_all_decisions(t)

        outputs.append({
            "variant_id": i,
            "transcript": t,
            "decisions": decisions
        })

    return outputs

In [ ]:
baseline = extract_all_decisions_itemwise(transcript)

In [ ]:
def compare_to_baseline(baseline, test_outputs):
    drift = []

    base_map = {d["item_id"]: d for d in baseline}

    for test in test_outputs:
        for d in test["decisions"]:
            item_id = d["item_id"]
            base = base_map.get(item_id)

            if not base:
                continue

            if d["decision"] != base["decision"]:
                drift.append({
                    "variant_id": test["variant_id"],
                    "item_id": item_id,
                    "baseline": base["decision"],
                    "variant": d["decision"],
                    "confidence": d["confidence"]
                })

    return drift

In [ ]:
import re
outputs = test_llm_stability(variants)

In [ ]:
compare_to_baseline(baseline , outputs)

[]

In [ ]:
result = extract_all_decisions_itemwise(variants[19])

In [ ]:
print(result)

[{'item_id': 1, 'decision': 'YES', 'reasoning': 'Sprinkler heads are located at the top and bottom of the stairways.', 'confidence': 1.0, 'date': None, 'start_time': None, 'end_time': None}, {'item_id': 2, 'decision': 'YES', 'reasoning': 'annual inspection and testing for both the sprinkler system and standpipes are up to date', 'confidence': 1.0, 'date': 'N/A', 'start_time': None, 'end_time': None}, {'item_id': 3, 'decision': 'YES', 'reasoning': 'annual inspection and testing for both the sprinkler system and standpipes are up to date', 'confidence': 1.0, 'date': 'N/A', 'start_time': None, 'end_time': None}, {'item_id': 4, 'decision': 'YES', 'reasoning': 'The fire hydrant is indeed located within one hundred feet of the fire department connection', 'confidence': 1.0, 'date': None, 'start_time': None, 'end_time': None}, {'item_id': 5, 'decision': 'YES', 'reasoning': 'the sprinkler system and standpipe signs are clearly visible', 'confidence': 1.0, 'date': None, 'start_time': None, 'end